# Limpieza de datos

In [1]:
import pandas as pd
import polars as pl 
from pathlib import Path

In [2]:
# Configuración de rutas relativas
RAW_DIR = Path("../data/raw/")
PROCESSED_DIR = Path("../data/processed")

# Verificacion
print("Archivos disponibles para procesar:")
for file in RAW_DIR.iterdir():
    print(f"- {file.name}")

Archivos disponibles para procesar:
- DICCIONARIO DE DATOS EMS 23-24.xlsx
- DICCIONARIO DE DATOS.xlsx
- diccionario_de_datos_tr_endutih_hogares_anual_2024.csv
- diccionario_de_datos_tr_endutih_residentes_anual_2024.csv
- diccionario_de_datos_tr_endutih_usuarios2_anual_2024.csv
- diccionario_de_datos_tr_endutih_usuarios_anual_2024.csv
- diccionario_de_datos_tr_endutih_viviendas_anual_2024.csv
- MEDIA SUPERIOR 2019-2020.csv
- MEDIA SUPERIOR 2020-2021.csv
- MEDIA SUPERIOR 2021-2022.csv
- MEDIA SUPERIOR 2022-2023.csv
- MEDIA_SUPERIOR 2023-2024.csv
- metadatos_endutih_anual_2024.txt
- modelo_er_endutih_2024.jpg
- tc_endutih_entidad.csv
- tr_endutih_hogares_anual_2024.csv
- tr_endutih_residentes_anual_2024.csv
- tr_endutih_usuarios2_anual_2024.csv
- tr_endutih_usuarios_anual_2024.csv
- tr_endutih_viviendas_anual_2024.csv


In [3]:
# Carga del archivo 
ruta_hogares = RAW_DIR / "tr_endutih_hogares_anual_2024.csv"
df_hogares = pd.read_csv(ruta_hogares, encoding='latin-1', low_memory=False)

# Filtrar CVE_ENT (7=Chiapas, 12=Guerrero, 20=Oaxaca)
df_sur = df_hogares[df_hogares['CVE_ENT'].isin([7, 12, 20])].copy()

# Limpieza
for col in ['P4_1_1', 'P4_1_2']:
    df_sur[col] = df_sur[col].fillna(2).replace(2, 0)

# Creacion del Indicador
df_sur['tiene_pc'] = (df_sur['P4_1_1'] == 1) | (df_sur['P4_1_2'] == 1)
df_sur['tiene_pc'] = df_sur['tiene_pc'].astype(int)

# Calculo por estado
print("Porcentaje de hogares con PC/Laptop en la muestra:")
print(df_sur.groupby('CVE_ENT')['tiene_pc'].mean() * 100)

Porcentaje de hogares con PC/Laptop en la muestra:
CVE_ENT
7     44.030668
12    29.453944
20    45.464248
Name: tiene_pc, dtype: float64


In [1]:
# Enrutamiento 
RUTA_ORIGEN = Path("../data/raw")
RUTA_DESTINO = Path("../data/processed")
RUTA_DESTINO.mkdir(parents=True, exist_ok=True)

print(f"--- Escaneando Datos: {RUTA_ORIGEN.resolve()} ---")

# Buscar los archivos en la ruta 
archivos = list(RUTA_ORIGEN.glob('*SUPERIOR*.csv'))
datos_historicos = []

if not archivos:
    print("No encontró ningún CSV con la palabra 'SUPERIOR' en ../data/raw/")
else:
    print(f"[+] Se encontraron {len(archivos)} archivos. Iniciando...")
    
    for archivo in archivos:
        delimitador = '|' if '2024 2025' in archivo.name else ','
        año = archivo.name.replace('.csv', '').replace('MEDIA SUPERIOR ', '').replace('MEDIA_SUPERIOR ', '')
        
        try:
            df = pd.read_csv(archivo, encoding='latin-1', sep=delimitador, low_memory=False)
            df.columns = df.columns.str.upper()
            
            col_ent = 'ENTIDAD' if 'ENTIDAD' in df.columns else 'CVE_ENT'
            df_sur = df[df[col_ent].isin([7, 12, 20])].copy()
            
            df_sur['ALUMNOS'] = pd.to_numeric(df_sur['ALUMNOS'], errors='coerce').fillna(0)
            df_sur['EXISTENTES'] = pd.to_numeric(df_sur['EXISTENTES'], errors='coerce').fillna(0)
            
            df_sur['DESERTORES'] = df_sur['ALUMNOS'] - df_sur['EXISTENTES']
            df_sur['DESERTORES'] = df_sur['DESERTORES'].apply(lambda x: x if x > 0 else 0)
            
            resumen = df_sur.groupby(col_ent).agg(
                matricula_inicial=('ALUMNOS', 'sum'),
                sobrevivientes=('EXISTENTES', 'sum'),
                bajas_totales=('DESERTORES', 'sum')
            ).reset_index()
            
            resumen['ciclo_escolar'] = año
            resumen['tasa_abandono_pct'] = (resumen['bajas_totales'] / resumen['matricula_inicial']) * 100
            
            datos_historicos.append(resumen)
            
        except Exception as e:
            print(f"[x] Error en {archivo.name}: {e}")

    # 3. Consolidacion
    if datos_historicos:
        df_final = pd.concat(datos_historicos, ignore_index=True)
        df_final = df_final.sort_values(by=['ENTIDAD', 'ciclo_escolar'])
        
        archivo_salida = RUTA_DESTINO / "abandono_sureste.csv"
        df_final.to_csv(archivo_salida, index=False)
        
        print("\n=======================================================")
        print(f"ÉXITO: Archivo generado en {archivo_salida}")
        print("=======================================================\n")
        # Mostrar la realidad de Chiapas
        print(df_final[df_final['ENTIDAD'] == 7][['ENTIDAD', 'ciclo_escolar', 'tasa_abandono_pct']])

NameError: name 'Path' is not defined

In [2]:
import pandas as pd
import plotly.express as px
import json

# 1. Cargar datos
df = pd.read_csv("../data/processed/abandono_sureste.csv")
nombres = {7: "Chiapas", 12: "Guerrero", 20: "Oaxaca"}
df['Estado'] = df['ENTIDAD'].map(nombres)

# 2. Crear Gráfica de Tendencia (Líneas)
fig_lineas = px.line(df, x="ciclo_escolar", y="tasa_abandono_pct", color="Estado", markers=True, title="Abandono Escolar en el Suroeste")
# GUARDAR COMO HTML INDEPENDIENTE
fig_lineas.write_html("../data/processed/grafica_lineas.html", include_plotlyjs="cdn")

# 3. Crear el Mapa (Con el GeoJSON que descargaste, o gráfica de barras si no lo tienes)
df_mapa = df[df['ciclo_escolar'] == "2023-2024"]
fig_barras = px.bar(df_mapa, x="Estado", y="tasa_abandono_pct", color="Estado", title="Tasa Actual (23-24)")
fig_barras.write_html("../data/processed/grafica_barras.html", include_plotlyjs="cdn")

print("Gráficas exportadas con éxito para Quarto.")

Gráficas exportadas con éxito para Quarto.
